In [18]:
with open("ironchat_info.txt", "r") as f:
    text = f.read()
print(text)

# IronChat - Complete Project Documentation

## 1. Project Overview
IronChat is a modern, responsive, and elegant AI chatbot application designed to provide quick questions, helpful answers, and smooth conversational flows. It leverages the raw speed of Llama-3 through Groq's APIs to provide real-time streaming answers. IronChat acts as an advanced intelligent assistant that can answer general knowledge questions, perform live web searches for up-to-date information, and read uploaded documents (PDFs) to answer questions based on the user's private data.

## 2. Core Features
- **Blazing Fast AI Responses:** Powered by Llama-3 70b via Groq API for near-instant inference and low latency.
- **Premium UI/UX:** Built with Tailwind CSS v4, featuring a glassmorphic dashboard, responsive layout, dark/light mode toggles, and smooth micro-animations.
- **Secure Authentication:** Built-in user sign up, login, and session management using JWT access and refresh tokens, with passwords hashed via Ar

In [6]:
def chunking(text, size=500, overlap=50):
    chunks = []
    start = 1
    while start < len(text):
        end  = start + size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += size - overlap
    return chunks

chunks = chunking(text)



Chunk 0
 IronChat - Complete Project Documentation

## 1. Project Overview
IronChat is a modern, responsive, and elegant AI chatbot application designed to provide quick questions, helpful answers, and smooth conversational flows. It leverages the raw speed of Llama-3 through Groq's APIs to provide real-time streaming answers. IronChat acts as an advanced intelligent assistant that can answer general knowledge questions, perform live web searches for up-to-date information, and read uploaded documents (
Chunk 1
to-date information, and read uploaded documents (PDFs) to answer questions based on the user's private data.

## 2. Core Features
- **Blazing Fast AI Responses:** Powered by Llama-3 70b via Groq API for near-instant inference and low latency.
- **Premium UI/UX:** Built with Tailwind CSS v4, featuring a glassmorphic dashboard, responsive layout, dark/light mode toggles, and smooth micro-animations.
- **Secure Authentication:** Built-in user sign up, login, and session management

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2") 
vectors = model.encode(chunks)  

print(f"Number of vectors: {len(vectors)}")
print(f"Length of one vector: {len(vectors[0])}")
print(vectors)

In [26]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(
    url=settings.QDRANT_URL,
    api_key=settings.QDRANT_API_KEY,
)

print(len(vectors[0])) 

COLLECTION_NAME = "ironchat_docs"

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=vectors[i],
        payload={"text": chunks[i]}
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Uploaded {len(points)} points to Qdrant")

384


/var/folders/kn/7pgm4glj1bv1r1g2ng4bhpl00000gn/T/ipykernel_98339/271672080.py:13: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Uploaded 13 points to Qdrant


In [29]:
question = "What is IronChat?"
question_vector = model.encode(question).tolist()

response = client.query_points(collection_name=COLLECTION_NAME, query=question_vector, limit=3)

for r in response.points:
    print(f"Score: {r.score:.4f}")
    print(r.payload["text"])
    print("---")


Score: 0.7155
 IronChat - Complete Project Documentation

## 1. Project Overview
IronChat is a modern, responsive, and elegant AI chatbot application designed to provide quick questions, helpful answers, and smooth conversational flows. It leverages the raw speed of Llama-3 through Groq's APIs to provide real-time streaming answers. IronChat acts as an advanced intelligent assistant that can answer general knowledge questions, perform live web searches for up-to-date information, and read uploaded documents (
---
Score: 0.5815
n generate an accurate answer based on the document.

## 5. How the Web Search Pipeline Works
IronChat does not just rely on its pre-trained knowledge; it can search the live internet.
1. **Decision Phase:** When a user sends a message, a hidden system prompt asks the LLM: "Does this require a live web search? Answer yes or no."
2. **Search Phase:** If the LLM says yes, the backend uses the Tavily API to search the web for the query and retrieves the top results 